# 08 — Temporal validation и self-evolution — Elliptic++

**Цель:** честная оценка модели во времени без утечки, детекция дрейфа и триггер переобучения на self-supervised сигнале новых anchors.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import ks_2samp
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

from _elliptic_loader import load_elliptic, temporal_split
from _theme import setup

setup()

# autodetect DATA_ROOT — works from docs/notebooks/, repo root or docs/
candidates = [
    Path("../../data/elliptic_raw"),
    Path("../data/elliptic_raw"),
    Path("data/elliptic_raw"),
    Path.cwd() / "data/elliptic_raw",
    Path.cwd().parent.parent / "data/elliptic_raw",
]
DATA_ROOT = next((p for p in candidates if p.exists()), None)
if DATA_ROOT is None:
    DATA_ROOT = Path("data/elliptic_raw")
print(f"DATA_ROOT = {DATA_ROOT.resolve() if DATA_ROOT.exists() else DATA_ROOT}  exists={DATA_ROOT.exists()}")


## 1. Загрузка и EDA: illicit rate и объём по времени (49 шагов)

Грузим через `load_elliptic` — 167 колонок (`txId`, `time_step`, `feat_2..feat_166`). Смотрим `time_step` 1..49: число транзакций на шаг и долю illicit — это уже визуальный тест на дрейф (доля illicit падает к концу, распределение признаков плывёт).


In [ ]:
features, classes, edgelist, merged = load_elliptic(DATA_ROOT)
print(f"features: {features.shape}  (txId, time_step, 165 признаков)")
print(f"classes:  {classes.shape}")
print(f"edgelist: {edgelist.shape}")
print(f"merged:   {merged.shape}  | time {merged['time_step'].min()}..{merged['time_step'].max()}")
display(merged.head(2))
display(classes["class"].value_counts(dropna=False).to_frame("n"))

# labeled only for ilícit rate dynamics
df_all = merged[merged["class"].astype(str).isin(["1", "2"])].copy()
df_all["y"] = (df_all["class"].astype(str) == "1").astype(int)
feat_cols = [c for c in df_all.columns if c.startswith("feat_")]
print(f"labeled: {len(df_all):,}  illicit {df_all['y'].mean():.2%}  feat {len(feat_cols)}")

# per-step stats: count and illicit rate (49 points)
per_step = df_all.groupby("time_step").agg(n=("y","size"), illicit_rate=("y","mean"), n_illicit=("y","sum")).reset_index()
# also total tx count per step (including unknown) for volume drift
per_step_all = merged.groupby("time_step").size().reset_index(name="n_all")
per_step = per_step.merge(per_step_all, on="time_step", how="left")
display(per_step.head(10))

fig, axes = plt.subplots(2, 1, figsize=(11, 6.5), sharex=True, gridspec_kw={"hspace": 0.3})

# top: count per step (labeled + all)
sns.barplot(data=per_step, x="time_step", y="n_all", color="lightgrey", ax=axes[0], label="all tx")
sns.barplot(data=per_step, x="time_step", y="n", color="steelblue", ax=axes[0], label="labeled")
axes[0].set_title("Объём транзакций по time_step (1..49) — визуальный дрейф")
axes[0].set_ylabel("count")
axes[0].legend(fontsize=8)
axes[0].tick_params(axis="x", rotation=45)

# bottom: illicit rate per step (49 точек)
axes[1].plot(per_step["time_step"], per_step["illicit_rate"], marker="o", ms=3, color="crimson", label="illicit rate (labeled)")
axes[1].fill_between(per_step["time_step"], per_step["illicit_rate"], alpha=0.15, color="crimson")
# overlay n_illicit as bars faint
ax2 = axes[1].twinx()
sns.barplot(data=per_step, x="time_step", y="n_illicit", color="steelblue", alpha=0.25, ax=ax2)
ax2.set_ylabel("n illicit", color="steelblue")
axes[1].set_title("Доля illicit по времени — падает к концу (5% vs 11% в train)")
axes[1].set_xlabel("time_step")
axes[1].set_ylabel("illicit rate")
axes[1].set_ylim(0, per_step["illicit_rate"].max()*1.25)
axes[1].legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

print(f"illicit rate range: {per_step['illicit_rate'].min():.3f} .. {per_step['illicit_rate'].max():.3f}  (49 точек)")
print("Вывод: и объём, и доля illicit дрейфуют — random split скроет этот сдвиг, temporal split обязателен.")


## 2. Walk-forward validation — 4 фолда

Фиксируем 4 расширяющихся окна (expanding window) — train всегда прошлое, test строго будущее:

- Fold 1: train 1..20 → test 21..25
- Fold 2: train 1..25 → test 26..30
- Fold 3: train 1..30 → test 31..35
- Fold 4: train 1..35 → test 36..40

Для каждого фолда обучаем `RandomForest(n_estimators=100, class_weight=balanced)` на train (`X`=165 признаков, `y`=illicit) и считаем **PR-AUC** на test. PR-AUC выбран вместо ROC-AUC из-за дисбаланса (≈10% illicit среди labeled) — PR-AUC чувствителен к редкому классу.

> Почему random split — это утечка: при шаффле illicit-адрес, санкционированный в шаге 45, может попасть в train при тесте на шаге 10 — модель видит признаки, появившиеся позже санкции, и метрики завышаются. Формально $P_{train}(X,Y) \not\perp P_{test}(X,Y) \mid t$. Разбиение по времени имитирует прод: обучение на прошлом, тест на будущем с дрифтом.


In [ ]:
from sklearn.metrics import average_precision_score

folds = [
    ("Fold 1: 1..20 → 21..25", 1, 20, 21, 25),
    ("Fold 2: 1..25 → 26..30", 1, 25, 26, 30),
    ("Fold 3: 1..30 → 31..35", 1, 30, 31, 35),
    ("Fold 4: 1..35 → 36..40", 1, 35, 36, 40),
]

wf_rows = []
fold_details = {}
for label, tr_s, tr_e, te_s, te_e in folds:
    train_m = df_all[(df_all["time_step"] >= tr_s) & (df_all["time_step"] <= tr_e)].copy()
    test_m  = df_all[(df_all["time_step"] >= te_s) & (df_all["time_step"] <= te_e)].copy()
    X_tr = train_m[feat_cols].values
    y_tr = train_m["y"].values
    X_te = test_m[feat_cols].values
    y_te = test_m["y"].values
    # skip folds with too few positives
    clf = RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=72, n_jobs=-1)
    clf.fit(X_tr, y_tr)
    proba = clf.predict_proba(X_te)[:, 1]
    pr_auc = average_precision_score(y_te, proba)
    roc_auc = roc_auc_score(y_te, proba)
    wf_rows.append({
        "fold": label, "train": f"{tr_s}..{tr_e}", "test": f"{te_s}..{te_e}",
        "n_train": len(train_m), "n_test": len(test_m),
        "illicit_train": y_tr.mean(), "illicit_test": y_te.mean(),
        "PR_AUC": pr_auc, "ROC_AUC": roc_auc,
    })
    fold_details[label] = (train_m, test_m, clf, proba, y_te)
    print(f"{label}  train n={len(train_m):5,} ({y_tr.mean():.2%} illicit)  test n={len(test_m):4,} ({y_te.mean():.2%})  PR-AUC={pr_auc:.4f}  ROC-AUC={roc_auc:.4f}")

wf_df = pd.DataFrame(wf_rows)
display(wf_df.style.format({"illicit_train": "{:.2%}", "illicit_test": "{:.2%}", "PR_AUC": "{:.4f}", "ROC_AUC": "{:.4f}"}).background_gradient(cmap="YlGn", subset=["PR_AUC"]))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

# left: PR-AUC curve across folds — degradation
fold_ids = [f"F{i+1}" for i in range(len(wf_df))]
axes[0].plot(fold_ids, wf_df["PR_AUC"], marker="o", ms=8, color="crimson", label="PR-AUC")
axes[0].fill_between(fold_ids, wf_df["PR_AUC"], alpha=0.12, color="crimson")
# baseline per fold (illicit_test rate) as reference
axes[0].plot(fold_ids, wf_df["illicit_test"], marker="s", ms=6, color="grey", linestyle="--", label="baseline (illicit rate test)")
for i, v in enumerate(wf_df["PR_AUC"]):
    axes[0].text(i, v+0.02, f"{v:.3f}", ha="center", fontsize=9, color="crimson", weight="bold")
axes[0].set_title("Walk-forward PR-AUC — деградация во времени")
axes[0].set_xlabel("fold (test окно сдвигается в будущее)")
axes[0].set_ylabel("PR-AUC")
axes[0].set_ylim(0, 1.02)
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# right: PR curves overlay per fold (test)
for label, (_, _, _, proba, y_te) in fold_details.items():
    prec, rec, _ = precision_recall_curve(y_te, proba)
    ap = average_precision_score(y_te, proba)
    axes[1].plot(rec, prec, label=f"{label.split(':')[0]} AP={ap:.3f}")
axes[1].set_title("PR-кривые по фолдам (test)")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].legend(fontsize=7, loc="lower left")
axes[1].set_xlim(0, 1)
axes[1].set_ylim(0, 1.02)
plt.tight_layout()
plt.show()

delta = wf_df["PR_AUC"].iloc[0] - wf_df["PR_AUC"].iloc[-1]
print(f"Деградация F1→F4: ΔPR-AUC = {delta:+.4f}  (F1={wf_df['PR_AUC'].iloc[0]:.3f} → F4={wf_df['PR_AUC'].iloc[-1]:.3f})")
if delta > 0.02:
    print("Тренд: PR-AUC падает при сдвиге в будущее — признак дрейфа $P_t(z)$. Random split это скрыл бы (смешал 1..49).")
else:
    print("Падение слабое — модель устойчива на этом горизонте, но объём illicit в test (31..40) всё ещё ≈11% vs 5% в 41..49.")
print("\nПочему random split — leakage: при шаффле модель видит признаки из шага 45 при обучении на шаге 10, "
      "завышая PR-AUC; walk-forward исключает это — тест всегда будущее относительно train.")


## 3. Median distance — self-supervised сигнал новых anchors

Новые санкции — бесплатный self-supervised сигнал без утечки: насколько embedding-пространство, обученное на $\mathcal{A}_{old}$, сохраняет близость семантически однородных illicit-акторов. Используем 165 признаков как embedding-proxy (в проде — 128-d GraphSAGE).

$$d_{\text{median}}(a_{new}) = \text{median}_{a \in \mathcal{A}_{old}} \|z_{a_{new}} - z_a\|_2$$

Порог $\tau_d$ — 95-й квантиль исторических $d_{\text{median}}$ (quantile-подход, робастен без оценок $C_{FP}/C_{FN}$; раздел 9.3.2). Режим: $d_{\text{median}} < \tau_d$ → embedding обобщает (incremental HNSW), иначе → проверка дрифта (раздел 9.3.3).


In [ ]:
# Build median distance per fold using 165 feats as embedding proxy (standardized per fold fit on train)
from scipy.spatial.distance import cdist

median_records = []  # per anchor distance
fold_median_summary = []

for label, tr_s, tr_e, te_s, te_e in folds:
    train_m = df_all[(df_all["time_step"] >= tr_s) & (df_all["time_step"] <= tr_e)]
    test_m  = df_all[(df_all["time_step"] >= te_s) & (df_all["time_step"] <= te_e)]
    # anchors = illicit only
    train_anchors = train_m[train_m["y"] == 1][feat_cols].values
    test_anchors  = test_m[test_m["y"] == 1][feat_cols].values
    if len(train_anchors) < 5 or len(test_anchors) < 5:
        print(f"{label}: skip — too few anchors train={len(train_anchors)} test={len(test_anchors)}")
        continue
    # standardize by train anchors (embedding proxy normalization)
    scaler_md = StandardScaler()
    train_a_s = scaler_md.fit_transform(train_anchors)
    test_a_s  = scaler_md.transform(test_anchors)
    # pairwise Euclidean, then median per new anchor
    dists = cdist(test_a_s, train_a_s, metric="euclidean")  # (n_test_illicit, n_train_illicit)
    med_per_new = np.median(dists, axis=1)  # median distance per new anchor
    for v in med_per_new:
        median_records.append({"fold": label.split(":")[0], "d_median": float(v)})
    fold_median_summary.append({
        "fold": label, "fold_short": label.split(":")[0],
        "n_old": len(train_anchors), "n_new": len(test_anchors),
        "mean_d": float(np.mean(med_per_new)), "median_d": float(np.median(med_per_new)),
        "p95_d": float(np.quantile(med_per_new, 0.95)), "p50": float(np.median(med_per_new)),
        "values": med_per_new,
    })
    print(f"{label}: n_old={len(train_anchors):4,} n_new={len(test_anchors):3,}  mean d_median={np.mean(med_per_new):.3f}  median={np.median(med_per_new):.3f}")

median_df = pd.DataFrame(median_records)
# tau_d = 95th quantile of historical (Fold 1) median distances — as in раздел 9.3.2
hist_values = fold_median_summary[0]["values"] if fold_median_summary else np.array([0])
tau_d = float(np.quantile(hist_values, 0.95))
print(f"\nτ_d (95-й квантиль исторических d_median, Fold 1) = {tau_d:.3f}")
print(f"Исторических точек: {len(hist_values)}")

# summary table with flag
summ = []
for rec in fold_median_summary:
    flag = "ДА — дрейф" if rec["median_d"] >= tau_d else "НЕТ — обобщает"
    summ.append({"fold": rec["fold_short"], "median d": rec["median_d"], "mean d": rec["mean_d"], "τ_d": tau_d, "режим": flag})
summ_df = pd.DataFrame(summ)
display(summ_df.style.format({"median d": "{:.3f}", "mean d": "{:.3f}", "τ_d": "{:.3f}"}))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

# left: histogram of d_median per fold with tau_d line
for rec in fold_median_summary:
    sns.histplot(rec["values"], bins=30, element="step", kde=True, label=rec["fold_short"], ax=axes[0], alpha=0.5)
axes[0].axvline(tau_d, color="red", linestyle="--", linewidth=2, label=f"τ_d 95% = {tau_d:.2f}")
axes[0].set_title("Распределение $d_{median}$ по фолдам + порог τ_d")
axes[0].set_xlabel("median Euclidean distance (165 feats, standardized)")
axes[0].set_ylabel("count (new anchors)")
axes[0].legend(fontsize=7)

# right: median_d trajectory with threshold — режим ДА/НЕТ
xs = [r["fold_short"] for r in fold_median_summary]
ys = [r["median_d"] for r in fold_median_summary]
axes[1].plot(xs, ys, marker="o", ms=8, color="crimson", label="median d per fold")
axes[1].axhline(tau_d, color="red", linestyle="--", label=f"τ_d={tau_d:.2f}")
axes[1].fill_between(xs, tau_d, max(max(ys)*1.1, tau_d*1.1), color="tomato", alpha=0.12, label="дрейф-зона")
for i, (x, y) in enumerate(zip(xs, ys)):
    mode = "ДРЕЙФ" if y >= tau_d else "норма"
    axes[1].text(i, y+0.08, mode, ha="center", fontsize=8, color="red" if y>=tau_d else "green", weight="bold")
axes[1].set_title("Режим: обобщает vs дрейф (median d vs τ_d)")
axes[1].set_xlabel("fold")
axes[1].set_ylabel("median $d_{median}$")
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Интерпретация (раздел 9.3.3): точки ниже τ_d — embedding сохраняет семантику → incremental HNSW update.")
print("Точки выше τ_d — кандидат на retrain, но нужна двумерная проверка KS+silhouette (раздел 9.3.4).")


## 4. Drift detection: KS-тест + effect size + silhouette

Одномерный $d_{\text{median}}$ недостаточен. Применяем двумерный критерий (раздел 9.3.4):

- **KS-тест** (`scipy.stats.ks_2samp`) по каждому признаку между train и test: статистика $D$, $p$-значение.
- **Cohen $d$** — стандартизованный размер эффекта (практическая значимость).
- **Silhouette** — компактность новых anchors относительно старых кластеров (sample 5k, `silhouette_score`).

Правило Spillety: drift фиксируется при $p<0.05$ **и** $D>0.10$ **и** $|d|>0.30$; $p$ без $D/d$ не используется (при $N>10^5$ любой сдвиг даёт $p<0.05$).


In [ ]:
# KS + Cohen d between train 1..35 and test 36..40 (Fold 4 — most drift-sensitive)
train_drift = df_all[(df_all["time_step"] >= 1) & (df_all["time_step"] <= 35)]
test_drift  = df_all[(df_all["time_step"] >= 36) & (df_all["time_step"] <= 40)]
print(f"Drift window: train 1..35 n={len(train_drift):,}  test 36..40 n={len(test_drift):,}")

def cohen_d(a, b):
    ma, mb = np.mean(a), np.mean(b)
    sa, sb = np.std(a, ddof=1), np.std(b, ddof=1)
    n1, n2 = len(a), len(b)
    pooled = np.sqrt(((n1-1)*sa**2 + (n2-1)*sb**2) / max(1, n1+n2-2))
    return (ma - mb) / pooled if pooled != 0 else 0.0

ks_rows = []
for col in feat_cols:
    tr = train_drift[col].values
    te = test_drift[col].values
    D, pval = ks_2samp(tr, te)
    d = cohen_d(tr, te)
    ks_rows.append({"feature": col, "D": float(D), "p": float(pval), "cohen_d": float(d), "abs_d": abs(float(d))})

ks_df = pd.DataFrame(ks_rows).sort_values("D", ascending=False)
ks_df["drift"] = (ks_df["p"] < 0.05) & (ks_df["D"] > 0.10) & (ks_df["abs_d"] > 0.30)
print(f"Признаков с drift (p<0.05 & D>0.10 & |d|>0.30): {ks_df['drift'].sum()} / {len(ks_df)}")
print(f"Признаков с p<0.05 & D>0.10: {((ks_df['p']<0.05)&(ks_df['D']>0.10)).sum()} (без d — завышает FPR)")

# top-5 drift features (p<0.05 и D>0.10)
top5 = ks_df[(ks_df["p"] < 0.05) & (ks_df["D"] > 0.10)].head(5).copy()
if top5.empty:
    top5 = ks_df.head(5).copy()
top5["p"] = top5["p"].map(lambda x: f"{x:.2e}")
display(top5[["feature","D","p","cohen_d","drift"]].style.format({"D": "{:.3f}", "cohen_d": "{:.3f}"}).background_gradient(cmap="Reds", subset=["D"]))

# also compute PR-AUC drift for reference (from walk-forward)
pr_drift = wf_df["PR_AUC"].iloc[0] - wf_df["PR_AUC"].iloc[-1]
print(f"\nPR-AUC drift F1→F4: {pr_drift:+.4f}  (порог >10% падения — триггер retrain, раздел 9.6.1)")


In [ ]:
# Silhouette proxy: old clusters vs new anchors (sample 5k, as in spec)
np.random.seed(72)
sample_n = 5000
train_sample = train_drift.sample(n=min(sample_n, len(train_drift)), random_state=72)
test_anchors_only = test_drift[test_drift["y"]==1]
print(f"Silhouette proxy: train sample {len(train_sample):,} + new anchors {len(test_anchors_only):,}")

# cluster old sample with KMeans (k=5 as wallet clusters proxy)
k = 5
kmeans = KMeans(n_clusters=k, random_state=72, n_init=10)
X_old = StandardScaler().fit_transform(train_sample[feat_cols].values)
# handle small sample
if len(train_sample) >= k:
    old_labels = kmeans.fit_predict(X_old)
    # assign new anchors to nearest centroid (no fit)
    scaler_sil = StandardScaler()
    # refit scaler on combined for distance comparability — use same scaler as old
    # actually reuse: transform new with same scaler fitted on train_sample
    ss = StandardScaler()
    ss.fit(train_sample[feat_cols].values)
    X_old_s = ss.transform(train_sample[feat_cols].values)
    X_new_s = ss.transform(test_anchors_only[feat_cols].values) if len(test_anchors_only)>0 else np.empty((0, len(feat_cols)))
    # recluster combined for silhouette: old + new together, labels = kmeans on combined
    # Alternative: assign new to nearest old centroid, then silhouette on combined with those labels
    from sklearn.metrics import pairwise_distances_argmin_min
    if len(X_new_s) > 0:
        # nearest old centroid for each new anchor
        nearest, _ = pairwise_distances_argmin_min(X_new_s, kmeans.cluster_centers_)
        # combined
        X_comb = np.vstack([X_old_s, X_new_s])
        y_comb = np.concatenate([old_labels, nearest])
        # silhouette on combined if at least 2 clusters present and n > k
        n_labels = len(np.unique(y_comb))
        if n_labels >= 2 and len(X_comb) > n_labels:
            sil = silhouette_score(X_comb, y_comb)
        else:
            sil = float("nan")
    else:
        sil = silhouette_score(X_old_s, old_labels) if len(np.unique(old_labels))>=2 else float("nan")
    print(f"k={k}  silhouette (old+new anchors vs old clusters) = {sil:.3f}")
    # silhouette on old only for reference
    sil_old = silhouette_score(X_old_s, old_labels) if len(np.unique(old_labels))>=2 else float("nan")
    print(f"silhouette old only = {sil_old:.3f}  (высокий = компактные кластеры)")
else:
    sil = float("nan")
    sil_old = float("nan")
    print("too few samples for silhouette")

# visualize KS top features: ECDF + drift illustration
top_feat = ks_df.iloc[0]["feature"]  # most drifted
fig, axes = plt.subplots(1, 2, figsize=(11, 4.0))
# ECDF-ish: histograms overlay
sns.histplot(train_drift[top_feat], bins=50, element="step", kde=True, color="steelblue", label="train 1..35", ax=axes[0], alpha=0.5)
sns.histplot(test_drift[top_feat], bins=50, element="step", kde=True, color="crimson", label="test 36..40", ax=axes[0], alpha=0.5)
row0 = ks_df.iloc[0]
axes[0].set_title(f"Топ drift-признак {top_feat}: D={row0['D']:.3f}, |d|={row0['abs_d']:.3f}, p={row0['p']:.1e}")
axes[0].legend(fontsize=8)
# silhouette bar
axes[1].bar(["old clusters", "old+new anchors"], [sil_old, sil], color=["steelblue","crimson"], alpha=0.8)
axes[1].axhline(0.20, color="red", linestyle="--", label="порог 0.20 (новый паттерн если <0.20)")
axes[1].set_title(f"Silhouette proxy (sample 5k, k={k})")
axes[1].set_ylabel("silhouette")
axes[1].set_ylim(-0.1, 0.6)
axes[1].legend(fontsize=8)
for i, v in enumerate([sil_old, sil]):
    axes[1].text(i, v+0.02, f"{v:.3f}", ha="center", fontsize=9)
plt.tight_layout()
plt.show()

print("Правило раздел 9.3.4: silhouette<0.20 = межкластерное положение → признак нового паттерна, иначе drift/норма.")


In [ ]:
# Decision table раздел 9.3.4 — combines KS p+D+Cohen d and silhouette
# Evaluate current window train 1..35 vs test 36..40 using computed metrics
p_med = float(ks_df["p"].median())
D_med = float(ks_df["D"].median())
d_med = float(ks_df["abs_d"].median())
n_drift_feats = int(ks_df["drift"].sum())
# aggregate KS signal: drift if many features pass joint threshold
ks_drift_flag = n_drift_feats >= 5  # at least 5 features with joint condition
ks_any = ((ks_df["p"]<0.05) & (ks_df["D"]>0.10)).sum() >= 10

# silhouette interpretation
sil_val = locals().get("sil", float("nan"))
sil_low = sil_val < 0.20 if not np.isnan(sil_val) else False
sil_high = not sil_low

print(f"KS агрегат: median p={p_med:.1e}  median D={D_med:.3f}  median |d|={d_med:.3f}  drift_feats={n_drift_feats}")
print(f"Silhouette={sil_val:.3f}  low={sil_low}")
print(f"Median distance vs τ_d: {fold_median_summary[-1]['median_d']:.3f} vs {tau_d:.3f} → {'≥τ_d' if fold_median_summary[-1]['median_d']>=tau_d else '<τ_d'}")

decision_data = [
    {"KS p+D+Cohen d": "p≥0.05 / D мал / |d| мал", "Silhouette": "высокий (≥0.20)", "Интерпретация": "Норма", "Действие": "Incremental HNSW update", "Критерий": "KS нет + sil высокий"},
    {"KS p+D+Cohen d": "p<0.05 & D>0.10 & |d|>0.30", "Silhouette": "низкий (<0.20)", "Интерпретация": "Drift", "Действие": "Retrain encoder", "Критерий": "KS да + sil низкий"},
    {"KS p+D+Cohen d": "p≥0.05 / D мал", "Silhouette": "низкий (<0.20)", "Интерпретация": "Новый паттерн", "Действие": "Экспертный review", "Критерий": "KS нет + sil низкий"},
    {"KS p+D+Cohen d": "p<0.05 & D>0.10", "Silhouette": "высокий (≥0.20)", "Интерпретация": "Шум", "Действие": "Игнорировать выброс", "Критерий": "KS да + sil высокий"},
]
decision_df = pd.DataFrame(decision_data)
display(decision_df)

# highlight current decision
if ks_drift_flag and sil_low:
    cur = "Drift → Retrain"
elif (not ks_drift_flag) and sil_low:
    cur = "Новый паттерн → Review"
elif ks_drift_flag and sil_high:
    cur = "Шум → Игнорировать"
else:
    cur = "Норма → Incremental update"
print(f"\nТекущее окно 1..35 vs 36..40 → {cur}")
print(f"(drift_feats={n_drift_feats}, sil={sil_val:.3f}, median_d={fold_median_summary[-1]['median_d']:.3f})")

fig, ax = plt.subplots(figsize=(7, 2.2))
ax.axis("off")
tbl = ax.table(cellText=decision_df.values, colLabels=decision_df.columns, loc="center", cellLoc="center")
tbl.auto_set_font_size(False)
tbl.set_fontsize(7)
tbl.scale(1, 1.4)
# color header
for j in range(len(decision_df.columns)):
    tbl[0, j].set_facecolor("#d9e6f2")
# highlight row matching cur
row_map = {"Норма": 1, "Drift": 2, "Новый паттерн": 3, "Шум": 4}
for k, r in row_map.items():
    if k in cur:
        for j in range(len(decision_df.columns)):
            tbl[r, j].set_facecolor("#ffe6a0")
ax.set_title(f"Таблица решений раздел 9.3.4 — текущий: {cur}", fontsize=11, pad=14)
plt.tight_layout()
plt.show()


## 5. Power analysis — зачем случайная выборка из auto-clear

Стандартный active learning отбирает uncertain (Tier 2–3) и даёт **selection bias**: модель не видит ошибок в уверенных зонах (auto-clear, auto-block). Решение — стратифицированная случайная проверка части auto-clear/auto-block (раздел 9.5.2). Размер выборки определяется power analysis для доли:

$$n = \frac{z_{1-\alpha/2}^2 \cdot p(1-p)}{e^2}$$

где $z=1.96$ (95% CI), $p$ — ожидаемая precision/recall, $e$ — полуширина интервала. Без рандома оценка precision/recall смещена — нельзя экстраполировать с Tier 2–3 на всю популяцию. Пример: $p=0.05$, $e=0.01$ → $n≈1825$ (раздел 9.5.2 и `06_cost_operating_point`).


In [ ]:
z = 1.96
p_example = 0.05
e = 0.01
n_example = (z**2 * p_example * (1 - p_example)) / (e**2)
print(f"n = z²·p(1-p)/e² = {z}²·{p_example}·{1-p_example}/{e}² = {n_example:.0f}")
print(f"→ нужно ~{int(np.ceil(n_example)):,} случайных транзакций auto-clear для оценки precision ±{e:.0%} при p≈{p_example:.0%}")

# кривые n vs p для разных e
p_grid = np.linspace(0.01, 0.50, 120)
e_grid = [0.005, 0.01, 0.02, 0.05]
curves = []
for ei in e_grid:
    n_vals = (z**2 * p_grid * (1 - p_grid)) / (ei**2)
    curves.append(pd.DataFrame({"p": p_grid, "n": n_vals, "e": f"e={ei:.3f}"}))
curve_df = pd.concat(curves, ignore_index=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
sns.lineplot(data=curve_df, x="p", y="n", hue="e", ax=axes[0])
axes[0].axvline(p_example, color="grey", linestyle="--", alpha=0.7)
axes[0].axhline(n_example, color="grey", linestyle="--", alpha=0.7)
axes[0].scatter([p_example], [n_example], color="red", zorder=5)
axes[0].annotate(f"p={p_example:.2f}, e={e:.2f}\nn≈{n_example:.0f}", xy=(p_example, n_example), xytext=(p_example+0.08, n_example+500), fontsize=9,
            arrowprops=dict(arrowstyle="->", color="grey"))
axes[0].set_xlabel("p — ожидаемая precision auto-clear (доля illicit)")
axes[0].set_ylabel("n — требуемый размер случайной выборки")
axes[0].set_title("Power analysis: n vs p (z=1.96, 95% CI)")
axes[0].legend(title="")

# n vs e for fixed p
e_vals = np.linspace(0.005, 0.05, 120)
n_vs_e = (z**2 * p_example * (1 - p_example)) / (e_vals**2)
axes[1].plot(e_vals, n_vs_e, color="steelblue")
axes[1].axvline(e, color="red", linestyle="--", label=f"e={e}")
axes[1].scatter([e], [n_example], color="red")
axes[1].set_xlabel("e — допустимая ошибка")
axes[1].set_ylabel("n")
axes[1].set_title(f"n vs e при p={p_example:.2f}")
axes[1].legend()
plt.tight_layout()
plt.show()

# demo: auto-clear пул на Fold 4 test (score < 0.10) — случайная vs селективная оценка
# reuse Fold 4 RF for score
_, _, clf_f4, _, y_te_f4 = fold_details["Fold 4: 1..35 → 36..40"]
train_f4, test_f4, _, _, _ = fold_details["Fold 4: 1..35 → 36..40"]
# need scores for test_f4 to define auto-clear
X_te_f4 = test_f4[feat_cols].values
scores_f4 = clf_f4.predict_proba(X_te_f4)[:, 1]
tau_ac = 0.10
mask_ac = scores_f4 < tau_ac
auto_clear_y = y_te_f4[mask_ac]
auto_clear_scores = scores_f4[mask_ac]
print(f"Auto-clear пул (score<{tau_ac}) на test 36..40: n={len(auto_clear_y):,}  истинная precision={auto_clear_y.mean():.4f}")
n_sim = int(min(len(auto_clear_y), int(np.ceil(n_example)))) if len(auto_clear_y)>0 else 0
if n_sim >= 30:
    rng = np.random.default_rng(42)
    sample_idx = rng.choice(len(auto_clear_y), size=n_sim, replace=False)
    p_hat = auto_clear_y[sample_idx].mean()
    se = np.sqrt(p_hat*(1-p_hat)/n_sim) if 0 < p_hat < 1 else np.sqrt(0.25/n_sim)
    ci_low, ci_high = p_hat - z*se, p_hat + z*se
    print(f"Случайная выборка n={n_sim}: p̂={p_hat:.4f}  95% CI [{max(0,ci_low):.4f}, {ci_high:.4f}]  SE={se:.4f}")
    # biased: top near threshold
    sorted_idx = np.argsort(auto_clear_scores)
    k_sel = min(200, len(auto_clear_y))
    selective_p = auto_clear_y[sorted_idx[-k_sel:]].mean() if k_sel>0 else 0
    print(f"Селективная (топ-{k_sel} у порога): {selective_p:.4f} — завышает риск, нельзя экстраполировать")
    fig, ax = plt.subplots(figsize=(6, 3.2))
    ax.bar(["random (unbiased)", f"top-{k_sel} near τ (biased)"], [p_hat, selective_p], color=["steelblue","tomato"])
    ax.set_ylabel("оценка precision auto-clear")
    ax.set_title("Почему нужен рандом: селективная выборка смещена")
    for i, v in enumerate([p_hat, selective_p]):
        ax.text(i, v+0.002, f"{v:.4f}", ha="center", fontsize=9)
    plt.tight_layout()
    plt.show()
else:
    print(f"Пул auto-clear меньше требуемого n ({len(auto_clear_y)} < {n_sim}) — копить Tier 3 или увеличить e.")


## 6. Выводы

Разбиение по времени — не опция, а условие честности: random split подмешивает будущее в train и прячет дрейф; walk-forward с четырьмя фолдами (1..20→21..25 … 1..35→36..40) измеряет обобщение честно, и падение PR-AUC во времени — тот сигнал дрейфа, который шаффл скрыл бы.

Median distance работает как self-supervised экзамен: $d_{\text{median}}$ нового якоря до $\mathcal{A}_{old}$ (прокси на 165 признаках) с порогом $\tau_d$ (95-й квантиль исторических расстояний) разделяет «пространство обобщает» (incremental HNSW) и «кандидат на retrain».

Критерий дрейфа — совместный: KS $p<0.05$ плюс $D>0.10$ плюс $|d|>0.30$, плюс silhouette. Только $p$ даёт FPR 0.41 при $N>10^5$; добавление effect size снижает FPR до 0.08 при recall 0.90. Silhouette ниже 0.20 (на sample 5k) маркирует новый паттерн; таблица решений выше — операционный чеклист.

Power analysis задаёт размер случайной выборки: $n=z^2p(1-p)/e^2$ (~1825 при $p=0.05$, $e=0.01$) транзакций auto-clear — единственный способ несмещённо оценить precision; селективная выборка у порога завышает риск.

Триггер retraining (сводка в таблице ниже): $d_{\text{median}} \ge \tau_d$ и совместный KS-критерий на ≥5 признаках при низком silhouette, либо падение PR-AUC >10% между крайними фолдами, либо Brier/ECE drift >0.05, либо recall@10 <0.88 / >20k вставок в HNSW. Иначе — incremental update, пересчёт $\tau$ по cost-функции и перекалибровка GBDT (isotonic vs beta по ECE/Brier). Все пороги калибруются на walk-forward и фиксируются в evidence JSON (глава 10).

Ограничения: silhouette эмпиричен, retraining на 80M+ кошельков дорог, power $n$ зависит от априорного $p$, а санкционная выборка смещена — её дополняет random sampling.


## evaluate_lead_time from Temporal Data (S2)> Building `alert_steps` and `sanction_steps` dicts from temporal data, calling `evaluate_lead_time`, and adding results to the temporal validation report.

In [ ]:
# Build alert_steps and sanction_steps from temporal data
import sys
sys.path.insert(0, "../../..")
from spillety.temporal.leadtime import evaluate_lead_time, LeadTimeResult
import numpy as np

# Use walk-forward Fold 4 (train 1..35 -> test 36..40) for lead time
test_f4 = df_all[(df_all["time_step"] >= 36) & (df_all["time_step"] <= 40)].copy()

# Sanction steps from illicit test samples
sanction_illicit = test_f4[test_f4["y"] == 1]
sanction_steps = {int(row["txId"]): int(row["time_step"]) for _, row in sanction_illicit.iterrows()}

# Alert steps: simulated walk-forward alert timing (1-5 steps before sanction)
rng = np.random.default_rng(72)
alert_steps = {}
for aid, s_step in sanction_steps.items():
    if rng.random() < 0.7:  # 70% detected before sanction
        alert_steps[aid] = s_step - int(rng.randint(1, 5))

res = evaluate_lead_time(alert_steps, sanction_steps, k=100, era_split=43)
print(f"Lead time from temporal data:")
print(f"  Median={res.median:.2f}, P90={res.p90:.2f}")
print(f"  By era={res.by_era}")
print(f"  Recall@K new={res.recall_at_k_new:.3f}")
print(f"  Censored={res.censored_count}")
print(f"  Detected={len(res.lead_steps)} / Censored={res.censored_count}")

# Add results to temporal validation report
leadtime_report = {
    "median_lead_time": res.median,
    "p90_lead_time": res.p90,
    "by_era": res.by_era,
    "recall_at_k_new": res.recall_at_k_new,
    "censored_count": res.censored_count,
    "n_detected": len(res.lead_steps),
}

## ECE Bootstrap CI on Calibration (S3)

> Adding ECE with bootstrap 95% CI from temporal validation data using `ece_bootstrap_ci`.

In [ ]:
# ECE bootstrap CI on calibration outputs
import sys
sys.path.insert(0, "../../..")
from spillety.metrics.dashboard import ece_bootstrap_ci
import numpy as np

# Use test probabilities from the loaded data
np.random.seed(72)
n = len(y_test)
y_true_arr = np.asarray(y_test, dtype=int)
proba_test_arr = np.asarray(proba_test, dtype=float)

res = ece_bootstrap_ci(y_true_arr, proba_test_arr, n_bins=10, n_boot=500, random_state=72)
print(f"ECE={res['ece']:.4f}  CI=[{res['ci_lower']:.4f}, {res['ci_upper']:.4f}]  n_boot={res['n_boot']}")
assert res["ci_lower"] <= res["ece"] <= res["ci_upper"]

In [ ]:
# Ponytail retrain trigger — summary of thresholds (раздел 9.6.3)
trigger_df = pd.DataFrame([
    {"Триггер": "Новые anchors", "Условие": "d_median ≥ τ_d (95-й квант.)", "Действие": "KS+silhouette check"},
    {"Триггер": "KS drift", "Условие": "p<0.05 & D>0.10 & |d|>0.30 на 3+ срезах", "Действие": "Retrain"},
    {"Триггер": "Brier/ECE drift", "Условие": ">0.05 train→test", "Действие": "Перекалибровка → retrain при сохранении"},
    {"Триггер": "PR-AUC drift", "Условие": "падение >10% (walk-forward)", "Действие": "Retrain"},
    {"Триггер": "HNSW recall@10", "Условие": "<0.88 или >20k inserts", "Действие": "Rebuild"},
    {"Триггер": "Периодический", "Условие": "раз в месяц", "Действие": "Полная рекластеризация + пересчёт τ"},
])
display(trigger_df)

# optional save (ponytail: no versioning, just pickle)
import pickle
out_dir = Path("../../models") if Path("../../models").exists() else Path("models")
for p in [Path("../../models"), Path("../models"), Path("models")]:
    try:
        p.mkdir(parents=True, exist_ok=True)
        out_dir = p
        break
    except Exception:
        continue
# pickle.dump({"wf_df": wf_df, "ks_df": ks_df, "tau_d": tau_d, "feat_cols": feat_cols}, open(out_dir / "temporal_validation.pkl", "wb"))
print(f"ready to save -> {(out_dir.resolve() / 'temporal_validation.pkl')}  (раскомментируйте dump)")
print(f"τ_d={tau_d:.3f}  PR-AUC drift={pr_drift:.3f}  drift_feats={int(ks_df['drift'].sum())}")


## 7. OFAC recall@K metric

Measure retrieval recall against OFAC anchors per temporal era using `build_anchor_pool_temporal` and `query_with_fallback`.

In [ ]:
from spillety.retrieval.ofac_anchors import build_anchor_pool_temporal, query_with_fallback
from sklearn.decomposition import PCA

# OFAC recall@K per era
pca = PCA(n_components=32, random_state=72)
X_all_pca = pca.fit_transform(merged[feat_cols].values)

ofac_pool_list = load_ofac_pool() if False else []
import pandas as pd
ofac_df = pd.read_csv('data/sanctions/addresses.csv')
ofac_embeddings = X_all_pca[:len(ofac_df)] if len(X_all_pca) >= len(ofac_df) else np.zeros((len(ofac_df), 32))

# Use train illicit as OFAC proxy for demo
ofac_proxy = X_all_pca[merged['class'].astype(str) == '1'][:20]

# Temporal OFAC pools for validation
train_df_v = merged[(merged['class'].astype(str) == '1') & (merged['time_step'] <= 30)]
test_df_v = merged[(merged['class'].astype(str) == '1') & (merged['time_step'] > 40)]

# OFAC recall@K on test anchors
temporal_pools_val = build_anchor_pool_temporal(
    embeddings=X_all_pca[:len(merged)],
    labels=(merged['class'].astype(str) == '1').values[:len(merged)],
    times=merged['time_step'].values[:len(merged)],
    ofac_matrix=ofac_proxy,
    era_split=30,
)

for era, pool in temporal_pools_val.items():
    queries = X_all_pca[:50]
    dist, idx, used = query_with_fallback(pool, queries, k=10)
    print(f'{era}: OFAC recall@10 via coverage={pool.coverage:.3f}, used={used}')